In [1]:
from pathlib import Path
import os, gc, json, hashlib, time, platform, importlib.metadata
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib
from IPython.display import display
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
from sklearn.model_selection import train_test_split, StratifiedKFold

MODE = os.environ.get('S6E10_SINGLE_MODE', 'full')
assert MODE in ['full', 'smoke']
SMOKE = MODE == 'smoke'
THREADS = 4
SPLIT_SEED, TUNING_SEED, INNER_SEED = 20261005, 20261006, 20261006
INNER_FOLDS = 3 if SMOKE else 5
EARLY_STOPPING = 12 if SMOKE else 180
FEATURE_VIEW = 'classic_route'
TARGET_STRENGTH = 20.0
MODEL_PARAMETERS = {'objective': 'binary', 'metric': 'auc', 'n_estimators': 3500, 'subsample_freq': 1, 'random_state': 42, 'verbosity': -1, 'deterministic': True, 'force_col_wise': True, 'cat_smooth': 20.0, 'cat_l2': 10.0, 'importance_type': 'gain', 'num_leaves': 127, 'min_child_samples': 160, 'max_bin': 511, 'learning_rate': 0.025, 'subsample': 0.8, 'colsample_bytree': 0.6, 'reg_lambda': 2.0, 'reg_alpha': 0.0}
MODEL_PARAMETERS['n_jobs'] = THREADS
if SMOKE:
    MODEL_PARAMETERS['n_estimators'] = 60
OUTPUT = Path(os.environ.get('S6E10_SINGLE_OUTPUT_DIR',
    '/kaggle/working' if Path('/kaggle/working').is_dir() else 's6e10/single_lgbm_run'))
OUTPUT.mkdir(parents=True, exist_ok=True)
VERSIONS = {name: importlib.metadata.version(name) for name in
    ['numpy', 'pandas', 'scikit-learn', 'lightgbm', 'joblib']}
assert int(lgb.__version__.split('.')[0]) >= 4, 'Use LightGBM4 or later'
print('Mode:', MODE, '| selected feature view:', FEATURE_VIEW)
print('One final classifier; CPU threads:', THREADS, '| versions:', VERSIONS)

Mode: full | selected feature view: classic_route
One final classifier; CPU threads: 4 | versions: {'numpy': '2.1.3', 'pandas': '2.2.3', 'scikit-learn': '1.6.1', 'lightgbm': '4.7.0', 'joblib': '1.4.2'}


## Data

### 1. Find and validate the official CSVs

The ID is only for joining/export, never a model input. A wrong competition or duplicate
folder stops explicitly; missing files never trigger downloads. Missing arrival delays are
retained for LightGBM plus an explicit missing flag. No test targets are read.

In [4]:
def sha256_file(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()

# def find_competition_data():
#     override = os.environ.get('S6E10_SINGLE_DATA_DIR')
#     choices = [Path(override)] if override else [
#         Path('/kaggle/input/competitions/playground-series-s6e10'),
#         Path('/kaggle/input/playground-series-s6e10'),
#         Path('../Predicting Airline Satisfaction'), Path('s6e10/data'), Path('data')]
#     if not override and Path('/kaggle/input').is_dir():
#         choices += [path.parent for path in Path('/kaggle/input').rglob('train.csv')
#                     if 'playground-series-s6e10' in str(path.parent)]
#     choices = list(dict.fromkeys(path.resolve() for path in choices if all(
#         (path / name).is_file() for name in ['train.csv', 'test.csv', 'sample_submission.csv'])))
#     if len(choices) != 1:
#         raise FileNotFoundError('Attach only the S6E10 competition Input, or set '
#             f'S6E10_SINGLE_DATA_DIR. Found {len(choices)} complete folders: {choices}')
#     return choices[0]

def binary_labels(series):
    text = series.astype('string').str.strip().str.lower()
    labels = text.map({'true': 1, 'false': 0, '1': 1, '0': 0, '1.0': 1, '0.0': 0,
                       'satisfied': 1, 'neutral or dissatisfied': 0, 'dissatisfied': 0})
    assert labels.notna().all(), 'Unexpected target values'
    return labels.to_numpy(dtype=np.int8)

# DATA = find_competition_data()
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample = pd.read_csv('sample_submission.csv')
ID, TARGET = 'id', 'satisfaction'
assert list(sample) == [ID, TARGET]
assert train[ID].is_unique and test[ID].is_unique
assert not train[ID].isin(test[ID]).any()
assert test[ID].equals(sample[ID]) and len(sample) == len(test)
y = binary_labels(train[TARGET])
assert set(np.unique(y)) == {0, 1}
INPUT_HASHES = {name: sha256_file(name) for name in
                ['train.csv', 'test.csv', 'sample_submission.csv']}
OFFICIAL_ROWS = {'train': len(train), 'test': len(test)}
if SMOKE:
    keep, _ = train_test_split(np.arange(len(train)), train_size=10000,
                              stratify=y, random_state=SPLIT_SEED)
    train = train.iloc[keep].reset_index(drop=True)
    y = y[keep]
    test = test.head(1000).reset_index(drop=True)
    sample = sample.head(1000).reset_index(drop=True)
else:
    assert len(train) == 699635 and len(test) == 299844, 'Not the reviewed full S6E10 data'
print('Train:', train.shape, '| test:', test.shape, '| positive rate:', round(float(y.mean()), 5))
display(train.head(3))

Train: (699635, 23) | test: (299844, 22) | positive rate: 0.44357


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False


### 2. Row-wise service and delay features

These depend only on the current row. The selected feature view was chosen by our own
tuning experiment; not every extra feature helped. IDs/labels are excluded by an allowlist.
Feature definitions below are fully included, so no local Python helper is required on Kaggle.

In [5]:
"""Competition-only features for a single LightGBM; no prediction inputs."""
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

RATINGS = ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
           'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
           'Inflight entertainment', 'On-board service', 'Leg room service',
           'Baggage handling', 'Checkin service', 'Cleanliness']
CATEGORIES = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
CONTINUOUS = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
RAW_FEATURES = CATEGORIES + CONTINUOUS + RATINGS
TARGET_KEYS = [
    ('Flight Distance',), ('Age',), ('Flight Distance', 'Age'),
    ('Flight Distance', 'Class'), ('Flight Distance', 'Type of Travel'),
    ('Age', 'Class', 'Type of Travel', 'Customer Type'),
    ('Inflight wifi service', 'Online boarding'), ('Class', 'Type of Travel')]
DEVIATION_RATINGS = ['Inflight wifi service', 'Online boarding', 'Inflight entertainment', 'Seat comfort']
FEATURE_RECIPE = 'competition_only_service_route_inner_te_v1'


def row_features(raw):
    """Only current-row predictors; neither ID nor label is an input."""
    frame = raw[RAW_FEATURES].copy().reset_index(drop=True)
    for name in CATEGORIES:
        frame[name] = frame[name].astype('string').str.strip().fillna('__missing__')
    for name in CONTINUOUS + RATINGS:
        frame[name] = pd.to_numeric(frame[name], errors='raise').astype('float32')
    ratings = frame[RATINGS]
    # A zero rating is retained as a distinct value, never silently replaced.
    frame['service_mean'] = ratings.mean(axis=1)
    frame['service_std'] = ratings.std(axis=1)
    frame['service_min'] = ratings.min(axis=1)
    frame['service_max'] = ratings.max(axis=1)
    frame['service_zero_count'] = ratings.eq(0).sum(axis=1).astype('float32')
    frame['service_low_count'] = (ratings.gt(0) & ratings.le(2)).sum(axis=1).astype('float32')
    frame['service_high_count'] = ratings.ge(4).sum(axis=1).astype('float32')
    frame['digital_mean'] = frame[['Inflight wifi service', 'Ease of Online booking', 'Online boarding']].mean(axis=1)
    frame['comfort_mean'] = frame[['Seat comfort', 'Leg room service', 'Cleanliness', 'Inflight entertainment']].mean(axis=1)
    frame['digital_minus_comfort'] = frame.digital_mean - frame.comfort_mean
    departure, arrival = frame[CONTINUOUS[-2]], frame[CONTINUOUS[-1]]
    frame['arrival_delay_missing'] = arrival.isna().astype('float32')
    frame['delay_difference'] = arrival - departure
    frame['delay_sum'] = arrival + departure
    for name in CONTINUOUS[-2:]:
        frame[name + '__log1p'] = np.log1p(frame[name].clip(lower=0))
        frame[name + '__zero'] = frame[name].eq(0).astype('float32')
    frame['distance_mod10'] = frame['Flight Distance'] % 10
    frame['distance_mod100'] = frame['Flight Distance'] % 100
    frame['age_mod10'] = frame.Age % 10
    # Dual representation preserves exact synthetic value identity.
    for name in ['Flight Distance', 'Age']:
        frame[name + '__identity'] = frame[name].astype('string').fillna('__missing__')
    frame['class_travel_key'] = frame['Class'].str.cat(frame['Type of Travel'], sep='|')
    frame['customer_travel_key'] = frame['Customer Type'].str.cat(frame['Type of Travel'], sep='|')
    return frame

### 3. Fit-only profiles and cross-fitted target means

Flight distance is a **synthetic grouping key**, not a known physical route ID.
Its training-only profiles capture repeated-value structure. Inner target-encoding folds
train only lookup statistics, **not additional prediction models**. Unseen keys use the
fitting prior/global profile; unseen categories are native categorical missing values.

In [6]:
def target_keys(frame):
    result = {}
    for number, columns in enumerate(TARGET_KEYS):
        text = frame[list(columns)].astype('string').fillna('__missing__')
        key = text.iloc[:, 0]
        for name in columns[1:]:
            key = key.str.cat(text[name], sep='|')
        result[f'target_mean_{number}'] = key
    return result


def target_mapping(key, labels, strength):
    stats = pd.DataFrame({'key': key.to_numpy(), 'label': np.asarray(labels)}).groupby('key').label.agg(['sum', 'count'])
    prior = float(np.mean(labels))
    return (stats['sum'] + strength * prior) / (stats['count'] + strength), prior


def prepare_features(fit_raw, labels, other_raw, *, inner_folds=5, seed=20261006,
                     target_strength=30.0, profile_strength=30.0):
    """Fit every lookup on fit_raw; training target means are inner out-of-fold.

    other_raw receives these fixed maps only, never enters fitted counts,
    categories, profiles or target statistics. Returns a serializable state.
    """
    labels = np.asarray(labels, dtype=np.int8)
    assert len(fit_raw) == len(labels) and set(np.unique(labels)) == {0, 1}
    assert pd.Series(labels).value_counts().min() >= inner_folds
    frames = [row_features(fit_raw)] + [row_features(raw) for raw in other_raw]
    fit = frames[0]
    state = {'recipe': FEATURE_RECIPE, 'fit_rows': len(fit), 'target_strength': target_strength,
             'profile_strength': profile_strength, 'frequencies': {}, 'categories': {}, 'target_maps': {}}
    for name in CONTINUOUS:
        frequency = fit[name].value_counts(dropna=False, normalize=True)
        state['frequencies'][name] = frequency
        for frame in frames:
            frame[name + '__frequency'] = frame[name].map(frequency).fillna(0).astype('float32')
    route_columns = ['Age'] + RATINGS
    grouped = fit.groupby('Flight Distance')[route_columns]
    global_mean = fit[route_columns].mean()
    count = grouped.count()
    profile = (grouped.sum() + profile_strength * global_mean) / (count + profile_strength)
    route_count = fit.groupby('Flight Distance').size()
    state.update(route_profile=profile, route_global_mean=global_mean, route_count=route_count)
    for frame in frames:
        distance = frame['Flight Distance']
        frame['route_log_count'] = np.log1p(distance.map(route_count).fillna(0)).astype('float32')
        for name in route_columns:
            feature = 'route_mean_' + name
            frame[feature] = distance.map(profile[name]).fillna(global_mean[name]).astype('float32')
            if name in DEVIATION_RATINGS:
                frame['route_deviation_' + name] = frame[name] - frame[feature]
    keys = [target_keys(frame) for frame in frames]
    inner = list(StratifiedKFold(inner_folds, shuffle=True, random_state=seed).split(fit, labels))
    coverage = np.zeros(len(fit), dtype=np.int8)
    for _, valid in inner:
        coverage[valid] += 1
    assert (coverage == 1).all()
    for name, key in keys[0].items():
        encoded = np.full(len(fit), np.nan, dtype=np.float32)
        for train_part, valid_part in inner:
            assert not np.intersect1d(train_part, valid_part).size
            mapping, prior = target_mapping(key.iloc[train_part], labels[train_part], target_strength)
            encoded[valid_part] = key.iloc[valid_part].map(mapping).fillna(prior).to_numpy(dtype=np.float32)
        assert np.isfinite(encoded).all()
        frames[0][name] = encoded
        mapping, prior = target_mapping(key, labels, target_strength)
        state['target_maps'][name] = {'mapping': mapping, 'prior': prior}
        for other_keys, frame in zip(keys[1:], frames[1:]):
            frame[name] = other_keys[name].map(mapping).fillna(prior).astype('float32')
    categorical_columns = [name for name in frames[0] if not pd.api.types.is_numeric_dtype(frames[0][name])]
    for name in categorical_columns:
        levels = sorted(frames[0][name].astype('string').unique())
        state['categories'][name] = levels
        for frame in frames:
            frame[name] = pd.Categorical(frame[name], categories=levels)
    numeric_columns = [name for name in frames[0] if name not in categorical_columns]
    for frame in frames:
        frame[numeric_columns] = frame[numeric_columns].astype('float32').replace([np.inf, -np.inf], np.nan)
        assert not {'id', 'satisfaction'}.intersection(frame.columns)
        assert list(frame) == list(frames[0]) and len(frame) > 0
    state['feature_names'] = list(frames[0])
    state['categorical_columns'] = categorical_columns
    state['inner_folds'] = inner_folds
    state['inner_seed'] = seed
    return frames, state


"""Small target-safe feature ablations; no auxiliary estimators."""
import numpy as np

EXTRA_ROW_COLUMNS = ['digital_minus_comfort', 'Flight Distance__identity', 'Age__identity',
                     'class_travel_key', 'customer_travel_key', 'route_log_count']


def prepare_single_features(fit_raw, labels, other_raw, *, feature_view='extended',
                            inner_folds=5, seed=20261006):
    assert feature_view in ['extended', 'no_numeric_categories', 'classic_route']
    labels = np.asarray(labels)
    assert labels.ndim == 1 and set(np.unique(labels)) == {0, 1}, 'Expected binary labels before conversion'
    assert inner_folds >= 2, 'Inner target encoding needs at least two folds'
    classic = feature_view == 'classic_route'
    # Tiny positive profile strength reproduces unshrunk means at float32 precision.
    frames, state = prepare_features(fit_raw, labels, other_raw, inner_folds=inner_folds,
                                    seed=seed, target_strength=20. if classic else 30.,
                                    profile_strength=1e-12 if classic else 30.)
    remove = []
    if feature_view == 'no_numeric_categories':
        remove = ['Flight Distance__identity', 'Age__identity']
    elif classic:
        remove = EXTRA_ROW_COLUMNS + ['route_deviation_' + name for name in DEVIATION_RATINGS]
    columns = [name for name in frames[0] if name not in remove]
    frames = [frame[columns].copy() for frame in frames]
    state['feature_view'] = feature_view
    state['feature_names'] = columns
    state['categorical_columns'] = [name for name in state['categorical_columns'] if name in columns]
    return frames, state

"""Bounded v2 LightGBM features: our classic route recipe, configurable TE smoothing."""
import numpy as np

FEATURE_RECIPE_V2 = 'own_classic_route_target_smoothing_v2'


def prepare_v2_features(fit_raw, labels, other_raw, *, target_strength=20.0,
                        inner_folds=5, seed=20261006):
    """Use only fit rows for maps; each fit-row target mean is inner OOF."""
    labels = np.asarray(labels)
    assert labels.ndim == 1 and set(np.unique(labels)) == {0, 1}
    assert inner_folds >= 2 and target_strength > 0
    frames, state = prepare_features(fit_raw, labels, other_raw,
        inner_folds=inner_folds, seed=seed, target_strength=target_strength,
        profile_strength=1e-12)
    remove = EXTRA_ROW_COLUMNS + ['route_deviation_' + name for name in DEVIATION_RATINGS]
    columns = [name for name in frames[0] if name not in remove]
    frames = [frame[columns].copy() for frame in frames]
    state.update(recipe_v2=FEATURE_RECIPE_V2, feature_view='classic_route', feature_names=columns)
    state['categorical_columns'] = [name for name in state['categorical_columns'] if name in columns]
    assert len(columns) == 66 and not {'id', 'satisfaction'}.intersection(columns)
    return frames, state


### 4. Check feature ranges and create the fixed partitions

These are disjoint row splits, not repeated training/test prediction averages. The20%
check is withheld from fitting and stopping; only the20% tuning selects tree count.
Duplicate predictor rows are reported, not silently deleted or assigned new labels.

In [7]:
assert set(train) == set(RAW_FEATURES + [ID, TARGET]), 'Unexpected competition schema'
assert set(test) == set(RAW_FEATURES + [ID])
for frame in [train, test]:
    for name in RATINGS:
        values = pd.to_numeric(frame[name], errors='raise')
        assert values.dropna().between(0, 5).all(), name + ' rating outside0..5'
QUALITY = {'duplicate_train_ids': int(train[ID].duplicated().sum()),
           'duplicate_train_predictor_rows': int(train.duplicated(RAW_FEATURES).sum()),
           'train_null_counts': train[RAW_FEATURES].isna().sum().to_dict(),
           'test_null_counts': test[RAW_FEATURES].isna().sum().to_dict()}
print('Repeated predictor rows:', QUALITY['duplicate_train_predictor_rows'])
display(pd.DataFrame({'train_missing': train[RAW_FEATURES].isna().sum(),
                      'test_missing': test[RAW_FEATURES].isna().sum()}).query(
                      'train_missing > 0 or test_missing > 0'))
development_ids, check_ids = train_test_split(np.arange(len(train)), test_size=.2,
                                              stratify=y, random_state=SPLIT_SEED)
fitting_ids, tuning_ids = train_test_split(development_ids, test_size=.25,
    stratify=y[development_ids], random_state=TUNING_SEED)
assert not np.intersect1d(fitting_ids, tuning_ids).size
assert not np.intersect1d(development_ids, check_ids).size
assert np.array_equal(np.sort(np.r_[fitting_ids, tuning_ids, check_ids]), np.arange(len(train)))
print('Fitting / tuning / check rows:', len(fitting_ids), len(tuning_ids), len(check_ids))

Repeated predictor rows: 0


,train_missing,test_missing
Arrival Delay in Minutes,292,130


Fitting / tuning / check rows: 419781 139927 139927


### 5. Prepare features without check/test-label access

Only fitting labels enter this function. The fitting features cannot change when test
categories/values change. We do not use the test distribution to fit any preprocessing map.

In [8]:
preparation_start = time.time()
validation_frames, validation_state = prepare_v2_features(
    train.iloc[fitting_ids][RAW_FEATURES], y[fitting_ids],
    [train.iloc[tuning_ids][RAW_FEATURES], train.iloc[check_ids][RAW_FEATURES]],
    target_strength=TARGET_STRENGTH, inner_folds=INNER_FOLDS, seed=INNER_SEED)
assert validation_state['fit_rows'] == len(fitting_ids)
assert not {ID, TARGET}.intersection(validation_state['feature_names'])
print('Selected features:', len(validation_state['feature_names']),
      '| preparation seconds:', round(time.time()-preparation_start, 1))

Selected features: 66 | preparation seconds: 21.6


## Results

### 6. Train the selected LightGBM configuration

Early stopping watches tuning ROC-AUC only. No parameter search or foreign predictions
run here. This diagnostic fit is later replaced by a new full-data fit, never blended with it.

In [9]:
started = time.time()
validation_model = lgb.LGBMClassifier(**MODEL_PARAMETERS)
validation_model.fit(validation_frames[0], y[fitting_ids],
    eval_set=[(validation_frames[1], y[tuning_ids])], eval_metric='auc',
    callbacks=[lgb.early_stopping(EARLY_STOPPING, first_metric_only=True, verbose=False),
               lgb.log_evaluation(250)])
BEST_ROUNDS = int(validation_model.best_iteration_ or MODEL_PARAMETERS['n_estimators'])
fitting_seconds = time.time()-started
tuning_prediction = validation_model.predict_proba(validation_frames[1])[:, 1]
check_prediction = validation_model.predict_proba(validation_frames[2])[:, 1]
assert np.isfinite(check_prediction).all() and ((check_prediction >= 0) & (check_prediction <= 1)).all()
TUNING_AUC = float(roc_auc_score(y[tuning_ids], tuning_prediction))
CHECK_METRICS = {'ROC_AUC': float(roc_auc_score(y[check_ids], check_prediction)),
                 'Average_precision': float(average_precision_score(y[check_ids], check_prediction)),
                 'Log_loss': float(log_loss(y[check_ids], check_prediction))}
display(pd.DataFrame({'metric': list(CHECK_METRICS), 'single_model_check': list(CHECK_METRICS.values())}))
print('Tuning ROC-AUC:', TUNING_AUC, '| best rounds:', BEST_ROUNDS,
      '| diagnostic fit seconds:', round(fitting_seconds, 1))
print('This check score is for one60%-trained model, NOT for the full refit or an ensemble.')
heldout = pd.DataFrame({ID: train.iloc[check_ids][ID].to_numpy(),
                       TARGET: y[check_ids], 'prediction': check_prediction})
heldout.to_csv(OUTPUT / 'heldout_single_lightgbm.csv', index=False)
diagnostic_importance = pd.DataFrame({'feature': validation_state['feature_names'],
    'gain': validation_model.booster_.feature_importance(importance_type='gain')})
diagnostic_importance.to_csv(OUTPUT / 'single_lightgbm_importance.csv', index=False)

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[250]	valid_0's auc: 0.959582
[500]	valid_0's auc: 0.960064


,metric,single_model_check
0,ROC_AUC,0.960934
1,Average_precision,0.949776
2,Log_loss,0.224362


Tuning ROC-AUC: 0.9600875277584057 | best rounds: 529 | diagnostic fit seconds: 70.1
This check score is for one60%-trained model, NOT for the full refit or an ensemble.


### 7. Discard the diagnostic model; rebuild maps on all training rows

The final tree count is fixed before this refit. Inner cross-fitting still excludes each
fitting row's target from its own target-mean feature. Test maps use all train labels, not
test labels. Holding more data for evaluation does not reduce the final training population.

In [10]:
del validation_model, validation_frames, validation_state, tuning_prediction, check_prediction
gc.collect()
full_preparation_start = time.time()
final_frames, final_state = prepare_v2_features(train[RAW_FEATURES], y, [test[RAW_FEATURES]],
    target_strength=TARGET_STRENGTH, inner_folds=INNER_FOLDS, seed=INNER_SEED)
assert final_state['fit_rows'] == len(train)
print('Final train:', final_frames[0].shape, '| test:', final_frames[1].shape,
      '| preparation seconds:', round(time.time()-full_preparation_start, 1))

Final train: (699635, 66) | test: (299844, 66) | preparation seconds: 42.9


### 8. Fit exactly one final estimator and predict directly

One classifier, trained from scratch, provides every submitted probability. No previous
model is loaded and no prediction file enters training. CPU is deliberate for portability.

In [12]:
final_parameters = {**MODEL_PARAMETERS, 'n_estimators': BEST_ROUNDS}
final_fit_started = time.time()
final_model = lgb.LGBMClassifier(**final_parameters)
final_model.fit(final_frames[0], y)
FINAL_FIT_SECONDS = time.time()-final_fit_started
final_prediction = final_model.predict_proba(final_frames[1])[:, 1]
assert len(final_prediction) == len(test) and np.isfinite(final_prediction).all()
assert ((final_prediction >= 0) & (final_prediction <= 1)).all()
print('One final estimator trained on', len(train), 'rows | trees:', final_model.booster_.num_trees(),
      '| seconds:', round(FINAL_FIT_SECONDS, 1))

One final estimator trained on 699635 rows | trees: 529 | seconds: 71.6


### 9. Export and verify the one-model submission

Save our own newly trained booster and feature maps, then reload **that same exported
model** only to verify its predictions. This is export QA, not another fit or a pretrained
input. `single_lightgbm_preprocessing.joblib` is locally created; never unpickle untrusted files.

In [13]:
FILENAME = 'SMOKE_NOT_FOR_KAGGLE_single_lightgbm.csv' if SMOKE else 'submission_single_lightgbm.csv'
exported = sample.copy()
exported[TARGET] = final_prediction
exported.to_csv(OUTPUT / FILENAME, index=False)
saved = pd.read_csv(OUTPUT / FILENAME)
assert list(saved) == [ID, TARGET] and saved[ID].equals(sample[ID]) and saved[ID].is_unique
assert len(saved) == len(test) and saved[TARGET].between(0, 1).all()
assert np.allclose(saved[TARGET], final_prediction, atol=1e-12, rtol=0)
MODEL_FILE = OUTPUT / ('SMOKE_single_lightgbm.txt' if SMOKE else 'single_lightgbm.txt')
final_model.booster_.save_model(str(MODEL_FILE))
joblib.dump(final_state, OUTPUT / 'single_lightgbm_preprocessing.joblib')
reloaded = lgb.Booster(model_file=str(MODEL_FILE))
model_reload_prediction = reloaded.predict(final_frames[1])
RELOAD_MAX_ERROR = float(np.max(np.abs(model_reload_prediction-final_prediction)))
assert RELOAD_MAX_ERROR < 1e-12
REPORT = {'competition': 'playground-series-s6e10', 'mode': MODE, 'algorithm': 'LightGBM',
    'feature_view': FEATURE_VIEW, 'feature_recipe': FEATURE_RECIPE_V2, 'target_strength': TARGET_STRENGTH, 'feature_count': len(final_state['feature_names']),
    'rows_train': len(train), 'rows_test': len(test), 'official_file_rows': OFFICIAL_ROWS,
    'diagnostic_fit_rows': len(fitting_ids), 'tuning_rows': len(tuning_ids), 'check_rows': len(check_ids),
    'tuning_auc': TUNING_AUC, 'single_diagnostic_check_metrics': CHECK_METRICS,
    'best_rounds': BEST_ROUNDS, 'parameters': final_parameters,
    'classifier_fits_in_notebook': 2, 'final_estimators_in_submission': 1,
    'prediction_averaging': False, 'foreign_predictions_used': False, 'teachers_used': False,
    'external_data_used': False, 'optuna_trials_run': 0, 'input_csv_sha256': INPUT_HASHES,
    'split_seed': SPLIT_SEED, 'tuning_seed': TUNING_SEED, 'inner_seed': INNER_SEED,
    'inner_folds': INNER_FOLDS, 'data_quality': QUALITY, 'versions': VERSIONS,
    'diagnostic_fit_seconds': fitting_seconds, 'final_fit_seconds': FINAL_FIT_SECONDS,
    'model_reload_max_abs_error': RELOAD_MAX_ERROR,
    'submission_file': FILENAME, 'submission_sha256': sha256_file(OUTPUT/FILENAME),
    'model_file_sha256': sha256_file(MODEL_FILE), 'public_score': None, 'previous_public_score_user_reported': 0.96018,
    'limitations': ['Single check score belongs to diagnostic fit, not all-data refit.',
        '12 own local candidates over adaptive rounds; tuning/check reused and prior experience also matters.',
        'Random synthetic-row split is not a temporal/generalization guarantee.',
        'No public/private score improvement is established for this new standalone model.']}
(OUTPUT/'single_lightgbm_report.json').write_text(json.dumps(REPORT, indent=2), encoding='utf-8')
print('Saved:', OUTPUT / FILENAME, '| one final model | reload max error:', RELOAD_MAX_ERROR)
if SMOKE:
    print('SMOKE ONLY: this CSV and these metrics are not competition results.')

Saved: s6e10\single_lgbm_run\submission_single_lightgbm.csv | one final model | reload max error: 0.0


## Phase 7A — 5-Fold Validation of the 0.96003 Feature Set

In [14]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import gc
import time
import lightgbm as lgb

# ============================================================
# PHASE 7A — 5-FOLD OUTER CV
# SAME 0.96003 FEATURE RECIPE
# ============================================================

N_OUTER_FOLDS = 5
OUTER_SEED = 20261006

outer_cv = StratifiedKFold(
    n_splits=N_OUTER_FOLDS,
    shuffle=True,
    random_state=OUTER_SEED
)

oof_pred = np.zeros(len(train), dtype=np.float64)
test_pred_folds = np.zeros((N_OUTER_FOLDS, len(test)), dtype=np.float64)

fold_results = []
fold_models = []

overall_start = time.time()

for fold, (train_idx, valid_idx) in enumerate(
    outer_cv.split(train, y), start=1
):

    print("\n" + "=" * 70)
    print(f"OUTER FOLD {fold}/{N_OUTER_FOLDS}")
    print("=" * 70)

    fold_start = time.time()

    # --------------------------------------------------------
    # Raw data
    # --------------------------------------------------------

    fold_train_raw = train.iloc[train_idx][RAW_FEATURES]
    fold_valid_raw = train.iloc[valid_idx][RAW_FEATURES]

    y_fold = y[train_idx]
    y_valid = y[valid_idx]

    # --------------------------------------------------------
    # TARGET-SAFE FEATURE PREPARATION
    # --------------------------------------------------------

    fold_frames, fold_state = prepare_v2_features(
        fold_train_raw,
        y_fold,
        [fold_valid_raw, test[RAW_FEATURES]],
        target_strength=TARGET_STRENGTH,
        inner_folds=INNER_FOLDS,
        seed=INNER_SEED
    )

    X_fold = fold_frames[0]
    X_valid = fold_frames[1]
    X_test = fold_frames[2]

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = lgb.LGBMClassifier(**MODEL_PARAMETERS)

    model.fit(
        X_fold,
        y_fold,
        eval_set=[(X_valid, y_valid)],
        eval_metric='auc',
        callbacks=[
            lgb.early_stopping(
                EARLY_STOPPING,
                first_metric_only=True,
                verbose=False
            )
        ]
    )

    best_iteration = int(
        model.best_iteration_ or MODEL_PARAMETERS['n_estimators']
    )

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    valid_pred = model.predict_proba(X_valid)[:, 1]
    fold_test_pred = model.predict_proba(X_test)[:, 1]

    fold_auc = roc_auc_score(y_valid, valid_pred)

    oof_pred[valid_idx] = valid_pred
    test_pred_folds[fold - 1] = fold_test_pred

    fold_time = time.time() - fold_start

    fold_results.append({
        'fold': fold,
        'auc': fold_auc,
        'best_iteration': best_iteration,
        'train_rows': len(train_idx),
        'valid_rows': len(valid_idx),
        'seconds': fold_time
    })

    print(f"Fold {fold} AUC       : {fold_auc:.9f}")
    print(f"Best iteration       : {best_iteration}")
    print(f"Fold time            : {fold_time:.1f}s")

    del (
        model,
        fold_frames,
        fold_state,
        X_fold,
        X_valid,
        X_test,
        valid_pred,
        fold_test_pred
    )

    gc.collect()


# ============================================================
# FINAL CV METRICS
# ============================================================

oof_auc = roc_auc_score(y, oof_pred)

fold_df = pd.DataFrame(fold_results)

mean_fold_auc = fold_df['auc'].mean()
std_fold_auc = fold_df['auc'].std(ddof=1)

test_prediction_5fold = test_pred_folds.mean(axis=0)

print("\n" + "=" * 70)
print("PHASE 7A — 5-FOLD RESULTS")
print("=" * 70)

display(fold_df)

print(f"\nMean Fold AUC : {mean_fold_auc:.9f}")
print(f"Std Fold AUC  : {std_fold_auc:.9f}")
print(f"OOF AUC       : {oof_auc:.9f}")
print(f"Previous      : 0.960030000")

print(
    f"\nOOF Delta     : {oof_auc - 0.96003:+.9f}"
)

print(
    f"Total time    : {(time.time() - overall_start)/60:.1f} min"
)


OUTER FOLD 1/5


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 1 AUC       : 0.960505725
Best iteration       : 546
Fold time            : 145.8s

OUTER FOLD 2/5


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 2 AUC       : 0.960236894
Best iteration       : 615
Fold time            : 155.8s

OUTER FOLD 3/5


...

Fold 4 AUC       : 0.960943772
Best iteration       : 562
Fold time            : 146.5s

OUTER FOLD 5/5


C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 5 AUC       : 0.960528930
Best iteration       : 647
Fold time            : 156.9s

PHASE 7A — 5-FOLD RESULTS


,fold,auc,best_iteration,train_rows,valid_rows,seconds
0,1,0.960506,546,559708,139927,145.782949
1,2,0.960237,615,559708,139927,155.823256
2,3,0.960550,591,559708,139927,148.568476
3,4,0.960944,562,559708,139927,146.537983
4,5,0.960529,647,559708,139927,156.850672



Mean Fold AUC : 0.960553136
Std Fold AUC  : 0.000252709
OOF AUC       : 0.960547974
Previous      : 0.960030000

OOF Delta     : +0.000517974
Total time    : 12.6 min


In [15]:
np.save(
    OUTPUT / 'champion_v2_5fold_oof.npy',
    oof_pred
)

np.save(
    OUTPUT / 'champion_v2_5fold_test.npy',
    test_prediction_5fold
)

fold_df.to_csv(
    OUTPUT / 'champion_v2_5fold_results.csv',
    index=False
)

np.save(
    OUTPUT / 'champion_v2_5fold_test_each_fold.npy',
    test_pred_folds
)

In [16]:
five_fold_report = {
    'experiment': 'champion_v2_5fold_outer_cv',
    'feature_recipe': FEATURE_RECIPE_V2,
    'feature_count': 66,
    'outer_folds': 5,
    'outer_seed': OUTER_SEED,
    'inner_folds': INNER_FOLDS,
    'inner_seed': INNER_SEED,
    'target_strength': TARGET_STRENGTH,
    'previous_holdout': 0.96003,
    'mean_fold_auc': float(mean_fold_auc),
    'std_fold_auc': float(std_fold_auc),
    'oof_auc': float(oof_auc),
    'best_iterations': fold_df['best_iteration'].tolist(),
    'fold_auc': fold_df['auc'].tolist()
}

(OUTPUT / 'champion_v2_5fold_report.json').write_text(
    json.dumps(five_fold_report, indent=2),
    encoding='utf-8'
)

606

## Phase 7B — XGBoost + CatBoost

### Phase 7B-1 - XGBoost

In [17]:
# ============================================================
# PHASE 7B-1 — XGBOOST 5-FOLD CV
# Exact same 66-feature pipeline as LightGBM
# ============================================================

import os
import gc
import time
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import xgboost as xgb

print("=" * 70)
print("PHASE 7B-1 — XGBOOST 5-FOLD CV")
print("=" * 70)

# ------------------------------------------------------------
# 1. CONFIG
# ------------------------------------------------------------

N_FOLDS = 5
SEED = 20261006
INNER_FOLDS = 5
INNER_SEED = 20261006
TARGET_STRENGTH = 20.0

OUTPUT_XGB = OUTPUT / "phase7b_xgb"
OUTPUT_XGB.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_XGB)

# ------------------------------------------------------------
# 2. SAME OUTER FOLDS
# ------------------------------------------------------------

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

folds = list(skf.split(train, y))

print("\nOuter folds:")
for fold, (train_idx, valid_idx) in enumerate(folds, 1):
    print(
        f"Fold {fold}: "
        f"train={len(train_idx):,} | "
        f"valid={len(valid_idx):,}"
    )

# Save fold indices so every future model uses EXACTLY
# the same validation rows.
np.savez(
    OUTPUT_XGB / "outer_folds.npz",
    **{
        f"fold_{i+1}_train": train_idx
        for i, (train_idx, _) in enumerate(folds)
    },
    **{
        f"fold_{i+1}_valid": valid_idx
        for i, (_, valid_idx) in enumerate(folds)
    }
)

# ------------------------------------------------------------
# 3. XGBOOST PARAMETERS
# ------------------------------------------------------------

XGB_PARAMS = {
    "n_estimators": 3000,
    "learning_rate": 0.03,

    "max_depth": 8,
    "min_child_weight": 5,

    "subsample": 0.85,
    "colsample_bytree": 0.85,

    "reg_alpha": 0.1,
    "reg_lambda": 1.0,

    "objective": "binary:logistic",
    "eval_metric": "auc",

    "tree_method": "hist",

    # Modern XGBoost:
    "device": "cuda",

    "random_state": SEED,

    "n_jobs": -1,
}

print("\nXGBoost parameters:")
for k, v in XGB_PARAMS.items():
    print(f"  {k}: {v}")

# ------------------------------------------------------------
# 4. STORAGE
# ------------------------------------------------------------

xgb_oof = np.zeros(len(train), dtype=np.float32)

xgb_test_folds = np.zeros(
    (N_FOLDS, len(test)),
    dtype=np.float32
)

fold_results = []

models = []

# ------------------------------------------------------------
# 5. FOLD LOOP
# ------------------------------------------------------------

overall_start = time.time()

for fold, (train_idx, valid_idx) in enumerate(folds, 1):

    print("\n" + "=" * 70)
    print(f"XGBOOST FOLD {fold}/{N_FOLDS}")
    print("=" * 70)

    fold_start = time.time()

    # --------------------------------------------------------
    # RAW DATA
    # --------------------------------------------------------

    fit_raw = train.iloc[train_idx][RAW_FEATURES]
    valid_raw = train.iloc[valid_idx][RAW_FEATURES]

    test_raw = test[RAW_FEATURES]

    y_fit = y[train_idx]
    y_valid = y[valid_idx]

    print(
        f"Fit rows   : {len(train_idx):,}\n"
        f"Valid rows : {len(valid_idx):,}\n"
        f"Test rows  : {len(test):,}"
    )

    # --------------------------------------------------------
    # TARGET-SAFE FEATURE ENGINEERING
    # EXACT SAME V2 RECIPE
    # --------------------------------------------------------

    prep_start = time.time()

    fold_frames, fold_state = prepare_v2_features(
        fit_raw,
        y_fit,
        [valid_raw, test_raw],
        target_strength=TARGET_STRENGTH,
        inner_folds=INNER_FOLDS,
        seed=INNER_SEED
    )

    X_fit = fold_frames[0]
    X_valid = fold_frames[1]
    X_test = fold_frames[2]

    print(
        f"Features: {X_fit.shape[1]} | "
        f"Preparation: {time.time() - prep_start:.1f}s"
    )

    assert X_fit.shape[1] == 66
    assert X_valid.shape[1] == 66
    assert X_test.shape[1] == 66

    # --------------------------------------------------------
    # XGBOOST CATEGORY HANDLING
    # --------------------------------------------------------

    categorical_columns = fold_state["categorical_columns"]

    # XGBoost native categorical handling.
    for col in categorical_columns:
        X_fit[col] = X_fit[col].astype("category")
        X_valid[col] = X_valid[col].astype("category")
        X_test[col] = X_test[col].astype("category")

    # Ensure identical category definitions.
    for col in categorical_columns:
        categories = X_fit[col].cat.categories

        X_valid[col] = X_valid[col].cat.set_categories(categories)
        X_test[col] = X_test[col].cat.set_categories(categories)

    print("Categorical features:", len(categorical_columns))

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model_start = time.time()

    model = xgb.XGBClassifier(
        **XGB_PARAMS,
        enable_categorical=True,
        early_stopping_rounds=100
    )

    model.fit(
        X_fit,
        y_fit,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )

    train_seconds = time.time() - model_start

    # --------------------------------------------------------
    # PREDICTIONS
    # --------------------------------------------------------

    valid_pred = model.predict_proba(X_valid)[:, 1]
    test_pred = model.predict_proba(X_test)[:, 1]

    # --------------------------------------------------------
    # VALIDATION SCORE
    # --------------------------------------------------------

    fold_auc = roc_auc_score(
        y_valid,
        valid_pred
    )

    best_iteration = getattr(
        model,
        "best_iteration",
        None
    )

    if best_iteration is not None:
        best_iteration += 1

    xgb_oof[valid_idx] = valid_pred
    xgb_test_folds[fold - 1] = test_pred

    fold_time = time.time() - fold_start

    fold_results.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "valid_rows": len(valid_idx),
        "auc": fold_auc,
        "best_iteration": best_iteration,
        "train_seconds": train_seconds,
        "total_fold_seconds": fold_time
    })

    print(
        f"\nFold {fold} AUC        : {fold_auc:.9f}"
    )
    print(
        f"Best iteration       : {best_iteration}"
    )
    print(
        f"Training time        : {train_seconds / 60:.2f} min"
    )
    print(
        f"Total fold time      : {fold_time / 60:.2f} min"
    )

    # --------------------------------------------------------
    # SAVE FOLD PREDICTIONS
    # --------------------------------------------------------

    fold_prediction = pd.DataFrame({
        "id": train.iloc[valid_idx][ID].to_numpy(),
        "target": y_valid,
        "prediction": valid_pred
    })

    fold_prediction.to_csv(
        OUTPUT_XGB / f"xgb_fold_{fold}_oof.csv",
        index=False
    )

    np.save(
        OUTPUT_XGB / f"xgb_fold_{fold}_test.npy",
        test_pred
    )

    # Save model
    model.save_model(
        OUTPUT_XGB / f"xgb_fold_{fold}.json"
    )

    # Save feature state
    joblib.dump(
        fold_state,
        OUTPUT_XGB / f"xgb_fold_{fold}_preprocessing.joblib"
    )

    models.append(model)

    # Free fold memory
    del (
        fit_raw,
        valid_raw,
        test_raw,
        X_fit,
        X_valid,
        X_test,
        fold_frames,
        fold_state,
        valid_pred,
        test_pred,
        model
    )

    gc.collect()

# ============================================================
# 6. OVERALL OOF RESULTS
# ============================================================

overall_time = time.time() - overall_start

oof_auc = roc_auc_score(
    y,
    xgb_oof
)

fold_df = pd.DataFrame(fold_results)

mean_auc = fold_df["auc"].mean()
std_auc = fold_df["auc"].std()

print("\n" + "=" * 70)
print("XGBOOST 5-FOLD RESULTS")
print("=" * 70)

print(f"\nFold mean AUC : {mean_auc:.9f}")
print(f"Fold std AUC  : {std_auc:.9f}")
print(f"OOF AUC       : {oof_auc:.9f}")
print(f"Total time    : {overall_time / 60:.2f} min")

display(fold_df)

# ------------------------------------------------------------
# 7. TEST PREDICTION AVERAGE
# ------------------------------------------------------------

xgb_test_prediction = xgb_test_folds.mean(axis=0)

print("\nTest prediction:")
print("Shape :", xgb_test_prediction.shape)
print("Min   :", xgb_test_prediction.min())
print("Max   :", xgb_test_prediction.max())
print("Mean  :", xgb_test_prediction.mean())

# ------------------------------------------------------------
# 8. SAVE OOF
# ------------------------------------------------------------

xgb_oof_df = pd.DataFrame({
    ID: train[ID].to_numpy(),
    TARGET: y,
    "xgb_oof_prediction": xgb_oof
})

xgb_oof_df.to_csv(
    OUTPUT_XGB / "xgb_5fold_oof.csv",
    index=False
)

np.save(
    OUTPUT_XGB / "xgb_oof.npy",
    xgb_oof
)

# ------------------------------------------------------------
# 9. SAVE TEST PREDICTION
# ------------------------------------------------------------

np.save(
    OUTPUT_XGB / "xgb_test_5fold_mean.npy",
    xgb_test_prediction
)

# ------------------------------------------------------------
# 10. KAGGLE SUBMISSION
# ------------------------------------------------------------

xgb_submission = sample.copy()

xgb_submission[TARGET] = xgb_test_prediction

XGB_SUBMISSION_FILE = (
    OUTPUT_XGB /
    "submission_xgb_5fold.csv"
)

xgb_submission.to_csv(
    XGB_SUBMISSION_FILE,
    index=False
)

# ------------------------------------------------------------
# 11. VALIDATE SUBMISSION
# ------------------------------------------------------------

saved = pd.read_csv(
    XGB_SUBMISSION_FILE
)

assert list(saved.columns) == [ID, TARGET]
assert len(saved) == len(test)
assert saved[ID].equals(sample[ID])
assert saved[ID].is_unique
assert saved[TARGET].between(0, 1).all()

print("\nSubmission validation: PASSED")

# ------------------------------------------------------------
# 12. SAVE FOLD RESULTS
# ------------------------------------------------------------

fold_df.to_csv(
    OUTPUT_XGB / "xgb_fold_results.csv",
    index=False
)

# ------------------------------------------------------------
# 13. SAVE SUMMARY JSON
# ------------------------------------------------------------

summary = {
    "competition": "playground-series-s6e10",
    "model": "XGBoost",
    "phase": "7B-1",

    "feature_recipe": FEATURE_RECIPE_V2,
    "feature_count": 66,

    "n_folds": N_FOLDS,
    "outer_seed": SEED,
    "inner_seed": INNER_SEED,
    "inner_folds": INNER_FOLDS,

    "target_strength": TARGET_STRENGTH,

    "fold_mean_auc": float(mean_auc),
    "fold_std_auc": float(std_auc),
    "oof_auc": float(oof_auc),

    "best_iterations": [
        None if pd.isna(x) else int(x)
        for x in fold_df["best_iteration"]
    ],

    "total_minutes": float(overall_time / 60),

    "test_prediction_mean": float(
        xgb_test_prediction.mean()
    ),

    "test_prediction_min": float(
        xgb_test_prediction.min()
    ),

    "test_prediction_max": float(
        xgb_test_prediction.max()
    ),

    "submission_file": str(
        XGB_SUBMISSION_FILE
    )
}

with open(
    OUTPUT_XGB / "xgb_5fold_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 14. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PHASE 7B-1 COMPLETE")
print("=" * 70)

print(f"""
XGBoost OOF AUC : {oof_auc:.9f}
Fold Mean AUC   : {mean_auc:.9f}
Fold Std        : {std_auc:.9f}

LGB Champion    : 0.960547974
XGB Result      : {oof_auc:.9f}

Test prediction : {xgb_test_prediction.shape}

Saved:
  {OUTPUT_XGB / "xgb_5fold_oof.csv"}
  {OUTPUT_XGB / "xgb_oof.npy"}
  {OUTPUT_XGB / "xgb_test_5fold_mean.npy"}
  {XGB_SUBMISSION_FILE}
  {OUTPUT_XGB / "xgb_fold_results.csv"}
  {OUTPUT_XGB / "xgb_5fold_summary.json"}
""")

PHASE 7B-1 — XGBOOST 5-FOLD CV
Output directory: s6e10\single_lgbm_run\phase7b_xgb

Outer folds:
Fold 1: train=559,708 | valid=139,927
Fold 2: train=559,708 | valid=139,927
Fold 3: train=559,708 | valid=139,927
Fold 4: train=559,708 | valid=139,927
Fold 5: train=559,708 | valid=139,927

XGBoost parameters:
  n_estimators: 3000
  learning_rate: 0.03
  max_depth: 8
  min_child_weight: 5
  subsample: 0.85
  colsample_bytree: 0.85
  reg_alpha: 0.1
  reg_lambda: 1.0
  objective: binary:logistic
  eval_metric: auc
  tree_method: hist
  device: cuda
  random_state: 20261006
  n_jobs: -1

XGBOOST FOLD 1/5
Fit rows   : 559,708
Valid rows : 139,927
Test rows  : 299,844
Features: 66 | Preparation: 36.3s
Categorical features: 4


C:\Users\Garv\anaconda3\Lib\site-packages\xgboost\core.py:569: UserWarning: [21:38:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



Fold 1 AUC        : 0.960531212
Best iteration       : 706
Training time        : 0.41 min
Total fold time      : 1.03 min

XGBOOST FOLD 2/5
Fit rows   : 559,708
Valid rows : 139,927
Test rows  : 299,844
Features: 66 | Preparation: 33.7s
Categorical features: 4

Fold 2 AUC        : 0.960128384
Best iteration       : 489
Training time        : 0.31 min
Total fold time      : 0.89 min

XGBOOST FOLD 3/5
Fit rows   : 559,708
Valid rows : 139,927
Test rows  : 299,844
Features: 66 | Preparation: 35.4s
Categorical features: 4

Fold 3 AUC        : 0.960329831
Best iteration       : 497
Training time        : 0.31 min
Total fold time      : 0.92 min

XGBOOST FOLD 4/5
Fit rows   : 559,708
Valid rows : 139,927
Test rows  : 299,844
Features: 66 | Preparation: 35.3s
Categorical features: 4

Fold 4 AUC        : 0.960854852
Best iteration       : 517
Training time        : 0.33 min
Total fold time      : 0.94 min

XGBOOST FOLD 5/5
Fit rows   : 559,708
Valid rows : 139,927
Test rows  : 299,844
Featur

,fold,train_rows,valid_rows,auc,best_iteration,train_seconds,total_fold_seconds
0,1,559708,139927,0.960531,706,24.531635,62.078140
1,2,559708,139927,0.960128,489,18.808623,53.599523
2,3,559708,139927,0.960330,497,18.867378,55.356933
3,4,559708,139927,0.960855,517,19.704549,56.158481
4,5,559708,139927,0.960527,655,21.793172,58.616632



Test prediction:
Shape : (299844,)
Min   : 0.005021745
Max   : 0.99873656
Mean  : 0.44408238

Submission validation: PASSED

PHASE 7B-1 COMPLETE

XGBoost OOF AUC : 0.960466103
Fold Mean AUC   : 0.960474326
Fold Std        : 0.000269908

LGB Champion    : 0.960547974
XGB Result      : 0.960466103

Test prediction : (299844,)

Saved:
  s6e10\single_lgbm_run\phase7b_xgb\xgb_5fold_oof.csv
  s6e10\single_lgbm_run\phase7b_xgb\xgb_oof.npy
  s6e10\single_lgbm_run\phase7b_xgb\xgb_test_5fold_mean.npy
  s6e10\single_lgbm_run\phase7b_xgb\submission_xgb_5fold.csv
  s6e10\single_lgbm_run\phase7b_xgb\xgb_fold_results.csv
  s6e10\single_lgbm_run\phase7b_xgb\xgb_5fold_summary.json



In [20]:
from pathlib import Path

ROOT = Path("s6e10")

matches = list(ROOT.rglob("*.npy"))

print("All .npy files containing 'oof':\n")

for p in matches:
    if "oof" in p.name.lower():
        print(p)

All .npy files containing 'oof':

s6e10\single_lgbm_run\champion_v2_5fold_oof.npy
s6e10\single_lgbm_run\phase7b_xgb\xgb_oof.npy


In [22]:
# ============================================================
# PHASE 7B-1.5 — LGB + XGB DIVERSITY / BLEND
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("PHASE 7B-1.5 — LGB + XGB DIVERSITY")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD OOF PREDICTIONS
# ------------------------------------------------------------

LGB_OOF_PATH = OUTPUT / "champion_v2_5fold_oof.npy"
XGB_OOF_PATH = OUTPUT / "phase7b_xgb" / "xgb_oof.npy"

lgb_oof = np.load(LGB_OOF_PATH)
xgb_oof = np.load(XGB_OOF_PATH)

print("\nLoaded:")
print("LGB:", LGB_OOF_PATH)
print("XGB:", XGB_OOF_PATH)

# ------------------------------------------------------------
# 2. SANITY CHECK
# ------------------------------------------------------------

assert len(lgb_oof) == len(y)
assert len(xgb_oof) == len(y)

assert np.isfinite(lgb_oof).all()
assert np.isfinite(xgb_oof).all()

print("\nShapes:")
print("LGB:", lgb_oof.shape)
print("XGB:", xgb_oof.shape)

# ------------------------------------------------------------
# 3. INDIVIDUAL AUC
# ------------------------------------------------------------

lgb_auc = roc_auc_score(y, lgb_oof)
xgb_auc = roc_auc_score(y, xgb_oof)

print("\n" + "=" * 70)
print("INDIVIDUAL OOF AUC")
print("=" * 70)

print(f"LightGBM : {lgb_auc:.9f}")
print(f"XGBoost  : {xgb_auc:.9f}")

# ------------------------------------------------------------
# 4. PREDICTION CORRELATION
# ------------------------------------------------------------

correlation = np.corrcoef(
    lgb_oof,
    xgb_oof
)[0, 1]

print("\nPrediction correlation:")
print(f"LGB ↔ XGB : {correlation:.9f}")

# ------------------------------------------------------------
# 5. PREDICTION DIFFERENCE
# ------------------------------------------------------------

difference = xgb_oof - lgb_oof

print("\nPrediction difference:")
print(f"Mean : {difference.mean():.9f}")
print(f"Std  : {difference.std():.9f}")
print(f"Min  : {difference.min():.9f}")
print(f"Max  : {difference.max():.9f}")

# ------------------------------------------------------------
# 6. WEIGHT SEARCH
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LGB + XGB BLEND SEARCH")
print("=" * 70)

results = []

# Fine search around all possible weights
weights = np.arange(0.00, 1.001, 0.01)

for xgb_weight in weights:

    lgb_weight = 1.0 - xgb_weight

    blend = (
        lgb_weight * lgb_oof +
        xgb_weight * xgb_oof
    )

    auc = roc_auc_score(
        y,
        blend
    )

    results.append({
        "lgb_weight": lgb_weight,
        "xgb_weight": xgb_weight,
        "auc": auc
    })

blend_results = pd.DataFrame(results)

blend_results = blend_results.sort_values(
    "auc",
    ascending=False
).reset_index(drop=True)

best = blend_results.iloc[0]

# ------------------------------------------------------------
# 7. RESULTS
# ------------------------------------------------------------

print("\nBest blend:")
print(
    f"LGB weight : {best['lgb_weight']:.2f}\n"
    f"XGB weight : {best['xgb_weight']:.2f}\n"
    f"AUC        : {best['auc']:.9f}"
)

print("\nTop 15 blends:")
display(blend_results.head(15))

# ------------------------------------------------------------
# 8. IMPROVEMENT
# ------------------------------------------------------------

improvement = best["auc"] - lgb_auc

print("\n" + "=" * 70)
print("BLEND IMPROVEMENT")
print("=" * 70)

print(f"LGB baseline : {lgb_auc:.9f}")
print(f"Best blend   : {best['auc']:.9f}")
print(f"Improvement  : {improvement:+.9f}")

# ------------------------------------------------------------
# 9. SAVE RESULTS
# ------------------------------------------------------------

BLEND_DIR = OUTPUT / "phase7b_xgb"

BLEND_DIR.mkdir(
    parents=True,
    exist_ok=True
)

blend_results.to_csv(
    BLEND_DIR / "lgb_xgb_blend_search.csv",
    index=False
)

best_oof = (
    best["lgb_weight"] * lgb_oof
    + best["xgb_weight"] * xgb_oof
)

np.save(
    BLEND_DIR / "lgb_xgb_best_blend_oof.npy",
    best_oof
)

# ------------------------------------------------------------
# 10. SAVE SUMMARY
# ------------------------------------------------------------

summary = {
    "lgb_auc": float(lgb_auc),
    "xgb_auc": float(xgb_auc),
    "prediction_correlation": float(correlation),

    "best_lgb_weight": float(best["lgb_weight"]),
    "best_xgb_weight": float(best["xgb_weight"]),
    "best_blend_auc": float(best["auc"]),

    "improvement_over_lgb": float(improvement)
}

import json

with open(
    BLEND_DIR / "lgb_xgb_blend_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print("\nSaved:")
print(BLEND_DIR / "lgb_xgb_blend_search.csv")
print(BLEND_DIR / "lgb_xgb_best_blend_oof.npy")
print(BLEND_DIR / "lgb_xgb_blend_summary.json")

PHASE 7B-1.5 — LGB + XGB DIVERSITY

Loaded:
LGB: s6e10\single_lgbm_run\champion_v2_5fold_oof.npy
XGB: s6e10\single_lgbm_run\phase7b_xgb\xgb_oof.npy

Shapes:
LGB: (699635,)
XGB: (699635,)

INDIVIDUAL OOF AUC
LightGBM : 0.960547974
XGBoost  : 0.960466103

Prediction correlation:
LGB ↔ XGB : 0.999030609

Prediction difference:
Mean : 0.000303305
Std  : 0.018969165
Min  : -0.449127546
Max  : 0.446258329

LGB + XGB BLEND SEARCH

Best blend:
LGB weight : 0.58
XGB weight : 0.42
AUC        : 0.960639714

Top 15 blends:


,lgb_weight,xgb_weight,auc
0,0.58,0.42,0.960640
1,0.59,0.41,0.960640
2,0.57,0.43,0.960640
3,0.60,0.40,0.960640
4,0.56,0.44,0.960639
5,0.61,0.39,0.960639
6,0.55,0.45,0.960639
7,0.62,0.38,0.960639
8,0.54,0.46,0.960639
9,0.63,0.37,0.960639



BLEND IMPROVEMENT
LGB baseline : 0.960547974
Best blend   : 0.960639714
Improvement  : +0.000091740

Saved:
s6e10\single_lgbm_run\phase7b_xgb\lgb_xgb_blend_search.csv
s6e10\single_lgbm_run\phase7b_xgb\lgb_xgb_best_blend_oof.npy
s6e10\single_lgbm_run\phase7b_xgb\lgb_xgb_blend_summary.json


### Phase 7B-2 — CatBoost

In [24]:
# ============================================================
# PHASE 7B-2 — CATBOOST 5-FOLD CV
# Exact same 66-feature pipeline
# Same outer folds as LGB + XGB
# Local RTX 3050
# ============================================================

import os
import gc
import time
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier

print("=" * 70)
print("PHASE 7B-2 — CATBOOST 5-FOLD CV")
print("=" * 70)

# ------------------------------------------------------------
# 1. CONFIG
# ------------------------------------------------------------

N_FOLDS = 5
SEED = 20261006

INNER_FOLDS = 5
INNER_SEED = 20261006

TARGET_STRENGTH = 20.0

OUTPUT_CAT = OUTPUT / "phase7b_catboost"
OUTPUT_CAT.mkdir(
    parents=True,
    exist_ok=True
)

print("Output directory:")
print(OUTPUT_CAT)

# ------------------------------------------------------------
# 2. RECREATE EXACT SAME OUTER FOLDS
# ------------------------------------------------------------

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

folds = list(
    skf.split(train, y)
)

print("\nOuter folds:")

for fold, (train_idx, valid_idx) in enumerate(folds, 1):

    print(
        f"Fold {fold}: "
        f"train={len(train_idx):,} | "
        f"valid={len(valid_idx):,}"
    )

# Save them for reproducibility
np.savez(
    OUTPUT_CAT / "outer_folds.npz",
    **{
        f"fold_{i+1}_train": train_idx
        for i, (train_idx, _) in enumerate(folds)
    },
    **{
        f"fold_{i+1}_valid": valid_idx
        for i, (_, valid_idx) in enumerate(folds)
    }
)

# ------------------------------------------------------------
# 3. CATBOOST PARAMETERS
# ------------------------------------------------------------

CAT_PARAMS = {
    "iterations": 3000,
    "learning_rate": 0.03,

    "depth": 8,

    "loss_function": "Logloss",
    "eval_metric": "AUC",

    "l2_leaf_reg": 5.0,

    "random_seed": SEED,

    "task_type": "GPU",

    # Important for local GPU stability
    "bootstrap_type": "Bayesian",

    "verbose": False,
}

print("\nCatBoost parameters:")

for key, value in CAT_PARAMS.items():
    print(f"  {key}: {value}")

# ------------------------------------------------------------
# 4. STORAGE
# ------------------------------------------------------------

cat_oof = np.zeros(
    len(train),
    dtype=np.float32
)

cat_test_folds = np.zeros(
    (N_FOLDS, len(test)),
    dtype=np.float32
)

fold_results = []

# ------------------------------------------------------------
# 5. FOLD LOOP
# ------------------------------------------------------------

overall_start = time.time()

for fold, (train_idx, valid_idx) in enumerate(folds, 1):

    print("\n" + "=" * 70)
    print(f"CATBOOST FOLD {fold}/{N_FOLDS}")
    print("=" * 70)

    fold_start = time.time()

    # --------------------------------------------------------
    # RAW DATA
    # --------------------------------------------------------

    fit_raw = train.iloc[train_idx][RAW_FEATURES]
    valid_raw = train.iloc[valid_idx][RAW_FEATURES]
    test_raw = test[RAW_FEATURES]

    y_fit = y[train_idx]
    y_valid = y[valid_idx]

    print(
        f"Fit rows   : {len(train_idx):,}\n"
        f"Valid rows : {len(valid_idx):,}\n"
        f"Test rows  : {len(test):,}"
    )

    # --------------------------------------------------------
    # SAME 66-FEATURE PIPELINE
    # --------------------------------------------------------

    prep_start = time.time()

    fold_frames, fold_state = prepare_v2_features(
        fit_raw,
        y_fit,
        [valid_raw, test_raw],
        target_strength=TARGET_STRENGTH,
        inner_folds=INNER_FOLDS,
        seed=INNER_SEED
    )

    X_fit = fold_frames[0].copy()
    X_valid = fold_frames[1].copy()
    X_test = fold_frames[2].copy()

    print(
        f"Features: {X_fit.shape[1]} | "
        f"Preparation: {time.time() - prep_start:.1f}s"
    )

    assert X_fit.shape[1] == 66
    assert X_valid.shape[1] == 66
    assert X_test.shape[1] == 66

    # --------------------------------------------------------
    # CATBOOST CATEGORICAL FEATURES
    # --------------------------------------------------------

    categorical_columns = fold_state["categorical_columns"]

    print(
        "Categorical features:",
        len(categorical_columns)
    )

    # CatBoost works most reliably with strings for
    # categorical columns.
    for col in categorical_columns:

        X_fit[col] = (
            X_fit[col]
            .astype("string")
            .fillna("__missing__")
            .astype(str)
        )

        X_valid[col] = (
            X_valid[col]
            .astype("string")
            .fillna("__missing__")
            .astype(str)
        )

        X_test[col] = (
            X_test[col]
            .astype("string")
            .fillna("__missing__")
            .astype(str)
        )

    categorical_indices = [
        X_fit.columns.get_loc(col)
        for col in categorical_columns
    ]

    print(
        "Categorical indices:",
        categorical_indices
    )

    # --------------------------------------------------------
    # TRAIN CATBOOST
    # --------------------------------------------------------

    model_start = time.time()

    model = CatBoostClassifier(
        **CAT_PARAMS,
        cat_features=categorical_indices,
        early_stopping_rounds=100
    )

    model.fit(
        X_fit,
        y_fit,
        eval_set=(X_valid, y_valid),
        verbose=False
    )

    train_seconds = time.time() - model_start

    # --------------------------------------------------------
    # PREDICTIONS
    # --------------------------------------------------------

    valid_pred = model.predict_proba(
        X_valid
    )[:, 1]

    test_pred = model.predict_proba(
        X_test
    )[:, 1]

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    fold_auc = roc_auc_score(
        y_valid,
        valid_pred
    )

    best_iteration = model.get_best_iteration()

    # CatBoost returns zero-based iteration
    if best_iteration is not None and best_iteration >= 0:
        best_iteration_display = best_iteration + 1
    else:
        best_iteration_display = None

    cat_oof[valid_idx] = valid_pred

    cat_test_folds[fold - 1] = test_pred

    fold_time = time.time() - fold_start

    fold_results.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "valid_rows": len(valid_idx),
        "auc": float(fold_auc),
        "best_iteration": best_iteration_display,
        "train_seconds": float(train_seconds),
        "total_fold_seconds": float(fold_time)
    })

    # --------------------------------------------------------
    # PRINT RESULT
    # --------------------------------------------------------

    print(
        f"\nFold {fold} AUC        : "
        f"{fold_auc:.9f}"
    )

    print(
        f"Best iteration       : "
        f"{best_iteration_display}"
    )

    print(
        f"Training time        : "
        f"{train_seconds / 60:.2f} min"
    )

    print(
        f"Total fold time      : "
        f"{fold_time / 60:.2f} min"
    )

    # --------------------------------------------------------
    # SAVE FOLD OOF
    # --------------------------------------------------------

    fold_prediction = pd.DataFrame({
        ID: train.iloc[valid_idx][ID].to_numpy(),
        TARGET: y_valid,
        "catboost_oof_prediction": valid_pred
    })

    fold_prediction.to_csv(
        OUTPUT_CAT / f"catboost_fold_{fold}_oof.csv",
        index=False
    )

    # --------------------------------------------------------
    # SAVE FOLD TEST
    # --------------------------------------------------------

    np.save(
        OUTPUT_CAT / f"catboost_fold_{fold}_test.npy",
        test_pred
    )

    # --------------------------------------------------------
    # SAVE MODEL
    # --------------------------------------------------------

    model.save_model(
        OUTPUT_CAT / f"catboost_fold_{fold}.cbm"
    )

    # --------------------------------------------------------
    # SAVE PREPROCESSING STATE
    # --------------------------------------------------------

    joblib.dump(
        fold_state,
        OUTPUT_CAT /
        f"catboost_fold_{fold}_preprocessing.joblib"
    )

    # --------------------------------------------------------
    # CLEAN MEMORY
    # --------------------------------------------------------

    del (
        fit_raw,
        valid_raw,
        test_raw,
        X_fit,
        X_valid,
        X_test,
        fold_frames,
        fold_state,
        valid_pred,
        test_pred,
        model
    )

    gc.collect()

# ============================================================
# 6. OVERALL OOF
# ============================================================

overall_time = time.time() - overall_start

cat_oof_auc = roc_auc_score(
    y,
    cat_oof
)

fold_df = pd.DataFrame(
    fold_results
)

cat_mean_auc = fold_df["auc"].mean()
cat_std_auc = fold_df["auc"].std()

print("\n" + "=" * 70)
print("CATBOOST 5-FOLD RESULTS")
print("=" * 70)

print(
    f"\nFold mean AUC : "
    f"{cat_mean_auc:.9f}"
)

print(
    f"Fold std AUC  : "
    f"{cat_std_auc:.9f}"
)

print(
    f"OOF AUC       : "
    f"{cat_oof_auc:.9f}"
)

print(
    f"Total time    : "
    f"{overall_time / 60:.2f} min"
)

display(
    fold_df
)

# ============================================================
# 7. TEST PREDICTION AVERAGE
# ============================================================

cat_test_prediction = (
    cat_test_folds.mean(axis=0)
)

print("\nTest prediction:")

print(
    "Shape :",
    cat_test_prediction.shape
)

print(
    "Min   :",
    cat_test_prediction.min()
)

print(
    "Max   :",
    cat_test_prediction.max()
)

print(
    "Mean  :",
    cat_test_prediction.mean()
)

# ============================================================
# 8. SAVE COMPLETE OOF
# ============================================================

cat_oof_df = pd.DataFrame({
    ID: train[ID].to_numpy(),
    TARGET: y,
    "catboost_oof_prediction": cat_oof
})

cat_oof_df.to_csv(
    OUTPUT_CAT / "catboost_5fold_oof.csv",
    index=False
)

np.save(
    OUTPUT_CAT / "catboost_oof.npy",
    cat_oof
)

# ============================================================
# 9. SAVE TEST PREDICTION
# ============================================================

np.save(
    OUTPUT_CAT / "catboost_test_5fold_mean.npy",
    cat_test_prediction
)

# ============================================================
# 10. CATBOOST SUBMISSION
# ============================================================

cat_submission = sample.copy()

cat_submission[TARGET] = (
    cat_test_prediction
)

CAT_SUBMISSION_FILE = (
    OUTPUT_CAT /
    "submission_catboost_5fold.csv"
)

cat_submission.to_csv(
    CAT_SUBMISSION_FILE,
    index=False
)

# ============================================================
# 11. VALIDATE SUBMISSION
# ============================================================

saved = pd.read_csv(
    CAT_SUBMISSION_FILE
)

assert list(saved.columns) == [
    ID,
    TARGET
]

assert len(saved) == len(test)

assert saved[ID].equals(
    sample[ID]
)

assert saved[ID].is_unique

assert saved[TARGET].between(
    0,
    1
).all()

print(
    "\nSubmission validation: PASSED"
)

# ============================================================
# 12. SAVE FOLD RESULTS
# ============================================================

fold_df.to_csv(
    OUTPUT_CAT /
    "catboost_fold_results.csv",
    index=False
)

# ============================================================
# 13. SAVE SUMMARY
# ============================================================

summary = {
    "competition": "playground-series-s6e10",
    "model": "CatBoost",
    "phase": "7B-2",

    "feature_recipe": FEATURE_RECIPE_V2,
    "feature_count": 66,

    "n_folds": N_FOLDS,

    "outer_seed": SEED,
    "inner_seed": INNER_SEED,
    "inner_folds": INNER_FOLDS,

    "target_strength": TARGET_STRENGTH,

    "fold_mean_auc": float(cat_mean_auc),
    "fold_std_auc": float(cat_std_auc),
    "oof_auc": float(cat_oof_auc),

    "best_iterations": [
        None
        if pd.isna(x)
        else int(x)
        for x in fold_df["best_iteration"]
    ],

    "total_minutes": float(
        overall_time / 60
    ),

    "test_prediction_mean": float(
        cat_test_prediction.mean()
    ),

    "test_prediction_min": float(
        cat_test_prediction.min()
    ),

    "test_prediction_max": float(
        cat_test_prediction.max()
    ),

    "submission_file": str(
        CAT_SUBMISSION_FILE
    )
}

with open(
    OUTPUT_CAT /
    "catboost_5fold_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# ============================================================
# 14. FINAL
# ============================================================

print("\n" + "=" * 70)
print("PHASE 7B-2 COMPLETE")
print("=" * 70)

print(
    f"""
CatBoost OOF AUC : {cat_oof_auc:.9f}
Fold Mean AUC    : {cat_mean_auc:.9f}
Fold Std         : {cat_std_auc:.9f}

LGB Champion     : 0.960547974
XGB              : 0.960466103
LGB + XGB        : 0.960639714
CatBoost         : {cat_oof_auc:.9f}

Test prediction  : {cat_test_prediction.shape}

Saved:
  {OUTPUT_CAT / "catboost_5fold_oof.csv"}
  {OUTPUT_CAT / "catboost_oof.npy"}
  {OUTPUT_CAT / "catboost_test_5fold_mean.npy"}
  {CAT_SUBMISSION_FILE}
  {OUTPUT_CAT / "catboost_fold_results.csv"}
  {OUTPUT_CAT / "catboost_5fold_summary.json"}
"""
)

PHASE 7B-2 — CATBOOST 5-FOLD CV
Output directory:
s6e10\single_lgbm_run\phase7b_catboost

Outer folds:
Fold 1: train=559,708 | valid=139,927
Fold 2: train=559,708 | valid=139,927
Fold 3: train=559,708 | valid=139,927
Fold 4: train=559,708 | valid=139,927
Fold 5: train=559,708 | valid=139,927

CatBoost parameters:
  iterations: 3000
  learning_rate: 0.03
  depth: 8
  loss_function: Logloss
  eval_metric: AUC
  l2_leaf_reg: 5.0
  random_seed: 20261006
  task_type: GPU
  bootstrap_type: Bayesian
  verbose: False

CATBOOST FOLD 1/5
Fit rows   : 559,708
Valid rows : 139,927
Test rows  : 299,844
Features: 66 | Preparation: 34.7s
Categorical features: 4
Categorical indices: [0, 1, 2, 3]


Default metric period is 5 because AUC is/are not implemented for GPU

KeyboardInterrupt


KeyboardInterrupt

